# Example 11.3.2 / 2: Name an Expression with a Defined Variable

Companion notebook for **11.3.2 Substitution and Expression Expansion** in the AMPL lab Documentation, section "Name an Expression with a Defined Variable". It runs the page's example from top to bottom: the model, the amplpy way (data.py, run.py), then the pure-AMPL way (data.dat, model.run). Solver: HiGHS.

## Install Dependencies
💡 TIP: Visit [portal.ampl.com](https://portal.ampl.com/user/ampl/license/list) or [dev.ampl.com](https://dev.ampl.com/ampl/install.html) for AMPL setup on Colab, Docker, Python, Windows, Linux, macOS

In [ ]:
# Install dependencies
%pip install -q amplpy

# Google Colab & Kaggle integration
from amplpy import AMPL, ampl_notebook
ampl = ampl_notebook(
    modules=["highs"],      # modules to install
    license_uuid="default") # license to use

## Name an Expression with a Defined Variable

A defined variable names a repeated nonlinear expression. The declaration `var y = expression;`
defines a variable through an equality-like relation; it does not freeze `y` at the expression's
value before optimization. AMPL substitutes the expression wherever the variable is used, so the
variable and its defining relation disappear from the solver view. This may reduce model size,
but it can also change sparsity, scaling, or the form a solver recognizes.

A defined unit cost $(10+4s)/(1+s)$ for shipment $s\ge2$: the model declares two variables and,
for `Cost`, one defining relation, but the solver receives one variable and no constraints.
`Cost` keeps its name for reporting: at the optimum $s=2$ it is $18/3=6$.

### model.mod
The model, written to a file that both ways of running it read.

In [2]:
%%writefile model.mod
var Ship >= 2, <= 10, := 3;               # shipment, bounded and started at 3
var Cost = (10 + 4 * Ship) / (1 + Ship);  # defined variable: unit cost of the shipment

minimize Total: Cost * Ship;  # total cost, unit cost times shipment

Writing model.mod


### amplpy: run.py

`ampl` is the `AMPL` object created above. Each run starts from a fresh session that has read `model.mod`; then `run.py` solves and reads the results.

In [3]:
# A fresh session that has read model.mod, as data.py and run.py assume
ampl.reset()
ampl.eval("reset options;")  # reset keeps the options
ampl.read("model.mod")

In [4]:
# run.py
ampl.solve(solver="ipopt", verbose=False)
print(ampl.solve_result, round(ampl.var["Ship"].value(), 6),
      round(ampl.var["Cost"].value(), 6), round(ampl.obj["Total"].value(), 6))
print([ampl.get_value(n) for n in ("_nvars", "_ncons", "_snvars", "_sncons")])

solved 2.0 6.0 12.0
[2, 1, 1, 0]
